In [1]:
import numpy as np
from numba import njit

from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

import scipy.sparse as sp
from scipy.spatial.distance import directed_hausdorff
import scipy.linalg as la

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings

import optuna

In [2]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

# Init

In [3]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [4]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [5]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [6]:
henon_scaler = StandardScaler()
henon_scaled = henon_scaler.fit_transform(henon_dataset)

In [7]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scattergl(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [8]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [9]:
def generate_henon_grid(all_experiments, cols=3, plot_height=400):
    total_plots = len(all_experiments)
    rows = (total_plots + cols - 1) // cols
    colors = ["white", "magenta"]

    # 1. Initialize the master subplot matrix
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"System #{i+1}" for i in range(total_plots)],
        horizontal_spacing=0.04,
        vertical_spacing=0.03,
    )

    for idx, data_list in enumerate(all_experiments):
        current_row = (idx // cols) + 1
        current_col = (idx % cols) + 1

        for i, data in enumerate(data_list):
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    marker=dict(color=colors[i % len(colors)], size=1),
                    showlegend=False,
                ),
                row=current_row,
                col=current_col,
            )

    fig.update_layout(
        height=plot_height * rows,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=10),
        margin=dict(t=80, b=40, l=40, r=40),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )

    return fig

# Bayesian Init

In [10]:
@njit(fastmath=True, cache=True)
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [11]:
@njit(fastmath=True, cache=True)
def compute_closed_states(
    steps_start,
    steps_end,
    Y_pred_scaled,
    X_pred,
    W_in,
    W_res,
    bias,
    alpha,
    W_out,
    W_bias,
    res_size,
    scaler_mean,
    scaler_std,
):
    for i in range(steps_start, steps_end):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        X_scaled = (X_pred[i, :res_size] - scaler_mean) / scaler_std
        Y_pred_scaled[i] = W_out @ X_scaled + W_bias
    return Y_pred_scaled

In [12]:
@njit(fastmath=True, cache=True)
def lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon):
    Q_test = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_test = np.zeros(2)
    for i in range(len(Y_test)):
        J = np.array([[-2.0 * a * Y_test[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_test)
        Q_test_raw, R = np.linalg.qr(Z)
        Q_test = np.ascontiguousarray(Q_test_raw)
        lyapunov_sums_test += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_test = lyapunov_sums_test / total_steps_after_henon

    Q_pred = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_pred = np.zeros(2)
    for i in range(len(Y_pred)):
        J = np.array([[-2.0 * a * Y_pred[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_pred)
        Q_pred_raw, R = np.linalg.qr(Z)
        Q_pred = np.ascontiguousarray(Q_pred_raw)
        lyapunov_sums_pred += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_pred = lyapunov_sums_pred / total_steps_after_henon

    le_loss = (lyapunov_exponent_test - lyapunov_exponent_pred) ** 2
    return le_loss[0]

In [13]:
def normalize_2d(data):
    min_vals = data.min(axis=0)
    max_vals = data.max(axis=0)
    return (data - min_vals) / (max_vals - min_vals)

In [14]:
# Basically initally the ideas was QR but that kinda cooks the sparsity
# Then Gram Shmidt but way too costly
# Now we take identity which is orthogonal and the least dense it can be basically
# Then we rotate using the rotation matrix which is also orthogonal and ends up giving an orthogonal output
# We also end up doing a DOK
# Went from DOK to just dense and numba


@njit(fastmath=True)
def random_sparse_orthogonal(dim, density, rng):
    Q = np.eye(dim)
    target_nnz = int(dim**2 * density)
    check_frequency = max(1, dim // 40)
    for _ in range(dim**2):
        i = rng.integers(0, dim)
        j = rng.integers(0, dim)
        while i == j:
            j = rng.integers(0, dim)

        theta = rng.uniform(0, 2 * np.pi)
        c, s = np.cos(theta), np.sin(theta)

        for k in range(dim):
            temp_i = Q[i, k]
            temp_j = Q[j, k]
            Q[i, k] = c * temp_i + s * temp_j
            Q[j, k] = -s * temp_i + c * temp_j

        if _ % check_frequency == 0:
            if np.count_nonzero(Q) >= target_nnz:
                break
    return Q


N = 200
rng = np.random.default_rng(42)
sparse_orthogonal = random_sparse_orthogonal(dim=N, density=0.1, rng=rng)

print("Sample slice of random float values:")
print(sparse_orthogonal[:5, :5].round(2))

print(f"\nTotal non-zero elements: {np.count_nonzero(sparse_orthogonal)}")
print(f"Actual matrix density: {np.count_nonzero(sparse_orthogonal) / (N**2):.4f}")

product = sparse_orthogonal @ sparse_orthogonal.T
identity = sp.eye(sparse_orthogonal.shape[0])
error_matrix = product - identity
orthogonality_error = np.linalg.norm(error_matrix, ord="fro")
print(f"Orthogonality Error: {orthogonality_error:.2e}")

Sample slice of random float values:
[[ 0.95  0.    0.    0.    0.  ]
 [-0.    0.01 -0.   -0.   -0.  ]
 [ 0.    0.   -0.11  0.    0.  ]
 [ 0.    0.    0.   -0.15  0.  ]
 [ 0.    0.    0.    0.    0.08]]

Total non-zero elements: 4041
Actual matrix density: 0.1010
Orthogonality Error: 2.77e-15


# Random Reservoir

In [ ]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
    mask = rng.random((res_size, res_size)) < sparsity
    W_res *= mask
    try:
        eigenvalues = la.eigvals(W_res)
    except Exception as e:
        print("Eigenval Issue")
        raise optuna.exceptions.TrialPruned()
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res * (spec_rad / largest_eigenvalue)

    noise = rng.normal(
        0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
    )
    X = compute_states(
        steps + transient_steps_reservoir - test_steps,
        henon_scaled[: -test_steps - tau_steps],
        res_size,
        W_in,
        W_res,
        bias,
        alpha,
        noise,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[transient_steps_reservoir:])
    Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]
    scaler_mean = scaler.mean_
    scaler_std = scaler.scale_

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train, Y_train)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train, Y_train)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    W_out = model.coef_
    W_bias = model.intercept_

    X_pred = np.zeros((test_steps, res_size))
    Y_pred_scaled = np.zeros((test_steps, out_size))
    Y_test = henon_dataset[-test_steps:]

    u = Y_train[-2]
    prev_state = X_train[-1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[0].reshape(1, -1))
    Y_pred_scaled[0] = model.predict(X_pred_scaled)

    u = Y_train[-1]
    prev_state = X_pred[0]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[1].reshape(1, -1))
    Y_pred_scaled[1] = model.predict(X_pred_scaled)

    compute_closed_states(
        0,
        test_steps,
        Y_pred_scaled,
        X_pred,
        W_in,
        W_res,
        bias,
        alpha,
        W_out,
        W_bias,
        res_size,
        scaler_mean,
        scaler_std,
    )

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

    return Y_test, Y_pred

In [107]:
res_size = 200
n_trials = 1000

In [108]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test[:75], Y_pred[:75])
    std = np.std(Y_test[:75])
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=n_trials, n_jobs=-1)

[Optuna] Processing Trial #999...

In [109]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 72/73...

In [110]:
generate_henon_grid(data, cols=5).show()

# Orthogonal Reservoir

In [30]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res = random_sparse_orthogonal(dim=res_size, density=sparsity, rng=rng)
    try:
        eigenvalues = la.eigvals(W_res)
    except Exception as e:
        print("Eigenval Issue")
        raise optuna.exceptions.TrialPruned()
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res * (spec_rad / largest_eigenvalue)

    noise = rng.normal(
        0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
    )
    X = compute_states(
        steps + transient_steps_reservoir - test_steps,
        henon_scaled[: -test_steps - tau_steps],
        res_size,
        W_in,
        W_res,
        bias,
        alpha,
        noise,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[transient_steps_reservoir:])
    Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]
    scaler_mean = scaler.mean_
    scaler_std = scaler.scale_

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train, Y_train)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train, Y_train)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    W_out = model.coef_
    W_bias = model.intercept_

    X_pred = np.zeros((test_steps, res_size))
    Y_pred_scaled = np.zeros((test_steps, out_size))
    Y_test = henon_dataset[-test_steps:]

    u = Y_train[-2]
    prev_state = X_train[-1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[0].reshape(1, -1))
    Y_pred_scaled[0] = model.predict(X_pred_scaled)

    u = Y_train[-1]
    prev_state = X_pred[0]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[1].reshape(1, -1))
    Y_pred_scaled[1] = model.predict(X_pred_scaled)

    compute_closed_states(
        0,
        test_steps,
        Y_pred_scaled,
        X_pred,
        W_in,
        W_res,
        bias,
        alpha,
        W_out,
        W_bias,
        res_size,
        scaler_mean,
        scaler_std,
    )

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

    return Y_test, Y_pred

In [98]:
res_size = 200
n_trials = 1000

In [100]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test[:75], Y_pred[:75])
    std = np.std(Y_test[:75])
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=n_trials, n_jobs=-1)

[Optuna] Processing Trial #999...

In [101]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 74/75...

In [102]:
generate_henon_grid(data, cols=5).show()

# Delay Line Reservoir

In [ ]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    injection_zone = 5
    W_in = np.zeros((res_size, in_size))
    W_in[:injection_zone, :] = rng.uniform(-0.5, 0.5, (injection_zone, in_size))
    W_res = np.diag(np.ones(res_size - 1) * spec_rad, k=-1)

    noise = rng.normal(
        0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
    )
    X = compute_states(
        steps + transient_steps_reservoir - test_steps,
        henon_scaled[: -test_steps - tau_steps],
        res_size,
        W_in,
        W_res,
        bias,
        alpha,
        noise,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[transient_steps_reservoir:])
    Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]
    scaler_mean = scaler.mean_
    scaler_std = scaler.scale_

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train, Y_train)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train, Y_train)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    W_out = model.coef_
    W_bias = model.intercept_

    X_pred = np.zeros((test_steps, res_size))
    Y_pred_scaled = np.zeros((test_steps, out_size))
    Y_test = henon_dataset[-test_steps:]

    u = Y_train[-2]
    prev_state = X_train[-1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[0].reshape(1, -1))
    Y_pred_scaled[0] = model.predict(X_pred_scaled)

    u = Y_train[-1]
    prev_state = X_pred[0]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[1].reshape(1, -1))
    Y_pred_scaled[1] = model.predict(X_pred_scaled)

    compute_closed_states(
        0,
        test_steps,
        Y_pred_scaled,
        X_pred,
        W_in,
        W_res,
        bias,
        alpha,
        W_out,
        W_bias,
        res_size,
        scaler_mean,
        scaler_std,
    )

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

    return Y_test, Y_pred

In [ ]:
res_size = 200
n_trials = 200

In [27]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[:, 0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[:, 1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test[:75], Y_pred[:75])
    std = np.std(Y_test[:75])
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=n_trials, n_jobs=-1)

[I 2026-06-29 23:42:20,549] A new study created in memory with name: no-name-1adc3a7d-4661-439c-85c5-20c745fb5081


[Optuna] Processing Trial #999...

In [28]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 38/39...

In [29]:
generate_henon_grid(data, cols=5).show()